# BFMNs from association and valence task

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add project root (parent of this scripts/ folder) to sys.path once
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Now import from ../src folder
from src.models_and_vars import load_env_vars, ensure_nltk_path, init_spacy_and_emo
from src.data_processing import subset_by_group_labels, build_association_df, filter_associations_by_occurrences
from src.valence import categorize_valence_multiple, plot_valence_distribution, fmn_valence_stats
from src.fmtools import (
    build_participants_bfmns,
    monitor_memory,
    draw_fmnts,
    make_emo_flowers,
    extract_bfmn_structural_features,
    all_fmns_features_df,
    verify_and_report_nodes
)
from src.io import export_df, export_whole_fmn_multiple_custom, import_fmnt_from_old_text_format

pd.set_option('display.max_rows', 100)

In [ ]:
# you do not need to run these download commands every time. nltk.download()
# saves these data packages to your local file system in the nltk_data directory.
# Once they are downloaded, they are available for NLTK to use in any session.

'''nltk.download('wordnet')
nltk.download('omw-1.4') # for italian and multilingual support, if needed
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # for word tokenization
nltk.download('averaged_perceptron_tagger') # for POS tagging
'''


# Environment and models settings

In [ ]:
# Set language once; everything else adapts
language = "italian"  # or "english"
env_file_path = "../env"

# Execute setup
load_env_vars(env_file_path, language=language)
ensure_nltk_path()

nlp, emo = init_spacy_and_emo(language)

# Also preload both models for explicit use
nlp_en, _ = init_spacy_and_emo("english")
nlp_it, _ = init_spacy_and_emo("italian")

# fmn and paths settings

In [ ]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
project_name = 'merged'
formatted_data_dir = 'formatting'
use_prelemmatized_brysbaert = True # A lot faster (highly suggested, but needs pre-lemmatized data)
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'bfmns'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica','ansia','stress','depressione','statistica','informatica','fisica','scienza','prestazione','esame','voto','valutazione','scuola','competizione','università']
# Lemmatization setting
lemmatize = False

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'highschool_south_unfiltered'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'

# If dfs_and_group_labels is longer than 1, merge the data from the different paths.
dfs_and_group_labels = []
df1 = pd.read_csv('../out/formatting/splitted_dfs/high_anxiety_high_school_gpt_oss.csv')
dfs_and_group_labels = [
    (df1, None)   # list of (df, group_labels)
]

# print all unique participants
all_participants = set()
for df, group_labels in dfs_and_group_labels:
    all_participants.update(df['participant_id'].unique())
print(f"All unique participants across datasets: {len(all_participants)}")
print(f"Unique participants labels: {all_participants}\n")

# Resolve output directories
if methods == {'aggregated'}:
    files_output_dir = f'{base_output_path}/{project_name}/aggregated/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/aggregated/{formatted_data_dir}/{fmn_type}'
if methods == {'individual'}:
    files_output_dir = f'{base_output_path}/{project_name}/individual/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/individual/{formatted_data_dir}/{fmn_type}'

## Run the pipeline

### Aggregated pipeline

In [ ]:
# Monitor initial memory usage
print("Starting pipeline...\n")

# Import the NetworkX graphs from txt files with custom valences
# Merge multiple edge Students files, filtered + unfiltered edges + student valences:
result = import_fmnt_from_old_text_format(
    files=[
        "../data/students_and_experts_stella_2019/Students_Unfiltered.txt",
        "../data/students_and_experts_stella_2019/ValenceStudentsUPD.txt"
    ],
    alpha=0.1,
    min_occurrences=3,
    direction_stat="mean",  # as requested
    directed=False
)

G_merged = result["graph"]
edges_df = result["edges"]
valence_df = result["valence_df"]
custom_valences = result["custom_valences"]

# Build the aggregated BFMN from the graph, keeping the custom valences
merged_fmn = emo.nxgraph_to_formamentis(G_merged)

monitor_memory()

# Export FMNs with custom valences
export_whole_fmn_multiple_custom(
    fmnts=[merged_fmn],
    save_path=files_output_dir,
    filename=f'{group_name}',
    custom_valences=custom_valences
)

# Load Brysbaert concreteness dataset
df_brysbaert = pd.read_csv('../data/brysbaert_concreteness_dataset_eng_ita/brysbaert_concreteness_dataset_eng_ita.csv')

# Load Brysbaert concreteness dataset
df_brysbaert = pd.read_csv('../data/brysbaert_concreteness_dataset_eng_ita/brysbaert_concreteness_dataset_eng_ita.csv')
# Draw and save semantic frame plots and emo flowers for each keyword
# for each keyword in keywords extract it, then draw the semantic
# frame plot, save it and show it in the notebook
layout = "edge_bundling"
if len(keywords) > 0:
    for keyword in keywords:
        semantic_frame = emo.extract_word_from_formamentis(merged_fmn, keyword)
        if not hasattr(semantic_frame, "vertices") or len(semantic_frame.vertices) == 0:
            print(f"Skipping '{keyword}': not in network or empty semantic frame.")
            continue
        out_dir = f'{base_output_path}/{files_output_dir}/visualizations'
        filename = f'{group_name}_{keyword}_{layout}_semantic_frame'
        draw_fmnts(
            fmnts={group_name: semantic_frame},
            out_dir=out_dir,
            layout=layout,
            highlight=[],
            thickness=1,
            ax=None,
            emo=emo,
            custom_valences=custom_valences,
            filename_base=filename,
        )
        make_emo_flowers(
            fmnts={f'{group_name}_{keyword}': semantic_frame},
            save_path=out_dir,
            filename=f'{group_name}_{keyword}_emo_flower',
            emo=emo,
            batch_size=1
        )

        if use_prelemmatized_brysbaert:
                # Analyze concreteness of the semantic frame (fast, pre-lemmatized datasets)
                try:
                    result = emo.analyze_semantic_frame_concreteness_fast(
                        semantic_frame,
                        seed=None,
                        save_json_path=f'{base_output_path}/{files_output_dir}/analysis/{group_name}_{keyword}_concreteness_analysis.json',
                        save_figure_path=f'{base_output_path}/{files_output_dir}/visualizations/{group_name}_{keyword}_concreteness_analysis.png',
                        plot=True
                    )
                    print(result['p_value'])
                    print(f"Concreteness analysis for '{keyword}' saved successfully.")
                except Exception as e:
                    print(f"Error analyzing concreteness for '{keyword}': {e}")
            
        else:
            # Analyze concreteness of the semantic frame
            try:
                result = emo.analyze_semantic_frame_concreteness(
                    semantic_frame,
                    df_brysbaert,
                    seed=None,
                    save_json_path=f'{base_output_path}/{files_output_dir}/analysis/{group_name}_{keyword}_concreteness_analysis.json',
                    save_figure_path=f'{base_output_path}/{files_output_dir}/visualizations/{group_name}_{keyword}_concreteness_analysis.png',
                    plot=True
                )
                print(result['p_value'])
                print(f"Concreteness analysis for '{keyword}' saved successfully.")
            except Exception as e:
                print(f"Error analyzing concreteness for '{keyword}': {e}")
        
monitor_memory()

valence_stats = fmn_valence_stats(merged_fmn, custom_valences)
print("Valence stats:", valence_stats)

print(G_merged)
print("Graph:", G_merged.nodes(data=True))
print("Custom valences:", custom_valences)

# Make a dictionary of the merged BFMN for further processing
if merged_fmn is not None:
    bfmns = {fmn_type: merged_fmn}

# Compute node_valence_maps for the merged BFMN
# Collect node valences separately (cannot attach to namedtuple)
node_val_maps = {}
node_val_maps[fmn_type] = {n: d.get("valence_cat") for n, d in G_merged.nodes(data=True)}

'''valence_stats = fmn_valence_stats(merged_fmn, custom_valences)
print("Valence stats:", valence_stats)

# Inspect the merged BFMN
inspect_fmn(
    fmn=merged_fmn,
    participant=group_name,
    all_features={group_name: valence_stats},
    emo=emo
)'''

# Check for presence of each keyword in the fmn
verify_and_report_nodes(
    fmns={group_name: merged_fmn},
    required=keywords
)

# Compute and export structural features for the aggregated FMN
aggregated_features = extract_bfmn_structural_features(
    fmnts={group_name: merged_fmn},
    save_path=f'{base_output_path}/{files_output_dir}/structural_features',
    emo=emo,
    batch_size=1,
    node_valence_maps=node_val_maps,
    persist=True
)

# Convert features to DataFrame and export
aggregated_features_df = all_fmns_features_df(aggregated_features, orient="participants")
export_df(aggregated_features_df, f'{base_output_path}/{files_output_dir}/structural_features', filename=f'{group_name}_structural_features')

aggregated_features_df = all_fmns_features_df(aggregated_features, orient="participants")
export_df(aggregated_features_df, f'{base_output_path}/{files_output_dir}/structural_features', filename=f'{group_name}_structural_features')

# Inspect the created fmnts
inspect_fmn(
    fmn=merged_fmn,
    participant=group_name,
    all_features=aggregated_features,
    emo=emo
)

### Verifying keywords presence in the networks

In [ ]:
# Verify presence of nodes "ansia", "matematica", "stress" and "depressione" in all BFMNs
check_summary = verify_and_report_nodes(
    fmns={fmn_type: merged_fmn},
    required=keywords,
)

print(check_summary)

In [ ]:
# Print number of nodes and edges (are the same as the ones imported: good!)
print(f"\nBFMN 2024 has {len(merged_fmn.vertices)} nodes and {len(merged_fmn.edges)} edges.")
print(f"BFMN 2025 has {len(merged_fmn.vertices)} nodes and {len(merged_fmn.edges)} edges.")

valence_stats_2025 = fmn_valence_stats(merged_fmn, custom_valences)
print("Valence stats 2025:", valence_stats_2025)